# 03 · Compartment review

A read-only overview of CellCharter domains followed by a focused look at one configurable domain. The composition and neighborhood panels use the tidy outputs from the spatial nearest-neighbor stage.

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

from spatial_workflow.config import load_config, resolve_path
from spatial_workflow.review import (
    available_compartments,
    compartment_composition,
    load_review_outputs,
    neighbor_fraction_matrix,
    plot_compartment_composition,
    plot_compartment_overview,
    plot_neighbor_fraction_heatmap,
    plot_spatial_domains,
    summarize_compartments,
)

CONFIG_PATH = REPO_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)
results_root = resolve_path(CONFIG_PATH, config["paths"]["results_root"])
input_h5ad = resolve_path(
    CONFIG_PATH, config["nncomp"]["input_h5ad"], root=results_root
)
output_dir = resolve_path(
    CONFIG_PATH, config["nncomp"]["output_dir"], root=results_root
)

tables = load_review_outputs(output_dir)
adata = ad.read_h5ad(input_h5ad, backed="r")

## Brief domain overview

In [ ]:
manifest = tables["manifest"]
display(
    pd.Series(
        {
            "cells": manifest["data"]["n_obs"],
            "samples": manifest["data"]["n_samples"],
            "conditions": ", ".join(manifest["data"]["conditions"]),
            "domains": len(manifest["data"]["compartments"]),
            "cell types": len(manifest["data"]["cell_types"]),
        },
        name="run",
    ).to_frame()
)

domain_overview = summarize_compartments(
    tables["compartment_abundance"]
)
display(domain_overview)
plot_compartment_overview(domain_overview)

In [ ]:
review_config = config.get("review", {})
sample_key = config["schema"]["sample_key"]
selected_sample = review_config.get("selected_sample")
if selected_sample is None:
    selected_sample = sorted(adata.obs[sample_key].astype(str).unique())[0]

plot_spatial_domains(
    adata,
    sample_col=sample_key,
    compartment_col=config["schema"]["spatial_domain_key"],
    spatial_key=config["schema"]["spatial_key"],
    sample=selected_sample,
)

## Selected domain

Set `review.selected_domain` in the YAML file. If it is null, the first real domain is selected; the synthetic whole-sample `all` bucket is not offered as a domain.

In [ ]:
domains = available_compartments(tables["compartment_abundance"])
selected_domain = review_config.get("selected_domain") or domains[0]
print(f"Selected domain: {selected_domain}")

composition = compartment_composition(
    tables["compartment_abundance"], selected_domain
)
display(composition)
plot_compartment_composition(
    composition, compartment=selected_domain
)

## Neighbor composition

The heatmap shows the mean fraction of each neighboring cell type around one source cell type, separately by condition. Configure the source with `review.selected_source_cell_type`.

In [ ]:
nn_summary = tables["neighbor_composition_summary"]
source_candidates = sorted(
    nn_summary.loc[
        nn_summary["compartment"].astype(str).eq(str(selected_domain)),
        "source_cell_type",
    ].astype(str).unique()
)
selected_source = (
    review_config.get("selected_source_cell_type") or source_candidates[0]
)
support_table = (
    nn_summary.loc[
        nn_summary["compartment"].astype(str).eq(str(selected_domain))
        & nn_summary["source_cell_type"].astype(str).eq(str(selected_source)),
        [
            "condition",
            "source_cell_type",
            "n_samples",
            "n_supporting_samples",
            "total_source_cells",
        ],
    ]
    .drop_duplicates(["condition", "source_cell_type"])
    .sort_values(["condition", "source_cell_type"])
    .reset_index(drop=True)
)
display(support_table)
fraction_matrix = neighbor_fraction_matrix(
    nn_summary,
    compartment=selected_domain,
    source_cell_type=selected_source,
)
display(fraction_matrix)
plot_neighbor_fraction_heatmap(
    fraction_matrix,
    title=f"Domain {selected_domain}: neighbors of {selected_source}",
)